# Parte 3 — Recommendations Engine

Demonstração do Recommendations Engine (`src/klike/recommender.py`): recebe uma campanha e gera recomendações acionáveis e quantificadas, com duas fontes de quantificação (what-if no modelo da Parte 2 + lift de performance com evidência, dos achados da Parte 1), priorizadas por impacto.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from klike import recommender as rec

pd.set_option("display.max_colwidth", 200)
df = rec.carregar_dados()
print(f"{len(df)} campanhas carregadas")

500 campanhas carregadas


## 2. Três campanhas contrastantes

- `KLK-0358` — TikTok, score baixo (22,4), sem hook, sem rosto, muito texto, formato quadrado.
- `KLK-0054` — LinkedIn, sem hook, sem rosto, texto médio, formato horizontal.
- `KLK-0024` — Meta, muito texto na tela, formato horizontal.

In [2]:
def mostrar_recomendacoes(campaign_id, n_min=3):
    campanha = df.loc[df["campaign_id"] == campaign_id].iloc[0]
    score_atual = rec.prever_score(campanha)
    print(f"=== {campaign_id} — {campanha['platform']} / {campanha['category']} ===")
    print(f"klike_score real: {campanha['klike_score']:.1f} | score previsto pelo modelo: {score_atual:.1f}\n")
    for i, r in enumerate(rec.recomendar(campanha, n_min=n_min), start=1):
        print(f"{i}. {r['texto']}\n")
    return campanha

In [3]:
_ = mostrar_recomendacoes("KLK-0358")

=== KLK-0358 — TikTok / SaaS ===
klike_score real: 22.4 | score previsto pelo modelo: 25.0



1. Adicionar um hook nos primeiros 3 segundos: na TikTok, criativos nessa condição têm CTR mediano +79% (faixa de incerteza: +25% a +151%; n=83 vs 93); na TikTok, criativos nessa condição têm fração assistida do vídeo mediano +34% (faixa de incerteza: +19% a +74%; n=73 vs 77). O modelo estima +16 pontos no Klike Score.

2. Incluir rosto humano no vídeo: na TikTok, criativos nessa condição têm CTR mediano +75% (faixa de incerteza: +27% a +120%; n=98 vs 78). O modelo estima +18 pontos no Klike Score.

3. Ajustar a proporção música/voz para mais equilibrada (~0.47): na TikTok, criativos nessa condição têm fração assistida do vídeo mediano +16% (faixa de incerteza: +1% a +38%; n=69 vs 56). impacto no score: desprezível segundo o modelo

4. Reduzir a quantidade de texto na tela de muito para médio: sem evidência de impacto em performance nos dados; impacto estimado apenas no score. O modelo estima +12 pontos no Klike Score.

5. Adicionar uma chamada para ação (CTA): sem evidência de impacto

In [4]:
_ = mostrar_recomendacoes("KLK-0054")

=== KLK-0054 — LinkedIn / App Install ===
klike_score real: 64.6 | score previsto pelo modelo: 50.3



1. Adicionar um hook nos primeiros 3 segundos: na LinkedIn, criativos nessa condição têm CTR mediano +119% (faixa de incerteza: +18% a +277%; n=45 vs 38). O modelo estima +15 pontos no Klike Score.

2. Reduzir a duração do vídeo para até 30s (mediana na plataforma: ~23s): no geral, criativos nessa condição têm CTR mediano +42% (faixa de incerteza: +6% a +84%; n=177 vs 56). o modelo não estima ganho de score para esta campanha específica

3. Incluir rosto humano no vídeo: sem evidência de impacto em performance nos dados; impacto estimado apenas no score. O modelo estima +11 pontos no Klike Score.



In [5]:
_ = mostrar_recomendacoes("KLK-0024")

=== KLK-0024 — Meta / Lead Gen ===
klike_score real: 43.1 | score previsto pelo modelo: 35.6



1. Reduzir a quantidade de texto na tela de muito para médio: na Meta, criativos nessa condição têm CTR mediano +91% (faixa de incerteza: +18% a +188%; n=105 vs 32); na Meta, criativos nessa condição têm fração assistida do vídeo mediano +26% (faixa de incerteza: +0% a +86%; n=90 vs 30). O modelo estima +20 pontos no Klike Score.

2. Incluir rosto humano no vídeo: na Meta, criativos nessa condição têm engajamento mediano +54% (faixa de incerteza: +22% a +103%; n=110 vs 114); na Meta, criativos nessa condição têm fração assistida do vídeo mediano +29% (faixa de incerteza: +10% a +44%; n=108 vs 105). O modelo estima +17 pontos no Klike Score.

3. Trocar o formato do vídeo para vertical: na Meta, criativos nessa condição têm engajamento mediano +44% (faixa de incerteza: +17% a +89%; n=102 vs 63). O modelo estima +2 pontos no Klike Score.

4. Adicionar um hook nos primeiros 3 segundos: na Meta, criativos nessa condição têm engajamento mediano +43% (faixa de incerteza: +10% a +95%; n=122 vs

## 3. Sanidade — todas as 500 campanhas

In [6]:
import time
import collections

t0 = time.time()
contagens = []
acoes_todas = []
for _, campanha in df.iterrows():
    recomendacoes = rec.recomendar(campanha, n_min=3)
    reais = [r for r in recomendacoes if r["acao"] is not None]
    contagens.append(len(reais))
    acoes_todas.extend(r["acao"] for r in reais)
print(f"tempo total: {time.time()-t0:.1f}s | cache de bootstrap: {len(rec._cache_lift)} entradas")

tempo total: 62.8s | cache de bootstrap: 134 entradas


In [7]:
print("distribuição do nº de recomendações por campanha:")
display(pd.Series(contagens).describe())

print("\nação mais recomendada (top 10):")
display(pd.Series(collections.Counter(acoes_todas)).sort_values(ascending=False).head(10).rename("n_campanhas"))

distribuição do nº de recomendações por campanha:


count    500.000000
mean       2.468000
std        1.188192
min        0.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        6.000000
dtype: float64


ação mais recomendada (top 10):


Adicionar um hook nos primeiros 3 segundos                                                    244
Incluir rosto humano no vídeo                                                                 231
Adicionar legendas                                                                            191
Adicionar uma chamada para ação (CTA)                                                         153
Reduzir a quantidade de texto na tela de muito para médio                                      91
Reduzir a quantidade de texto na tela de médio para pouco                                      69
Ajustar a proporção música/voz para mais equilibrada (~0.47)                                   62
Verificar se o vídeo tem legendas (não informado nesta campanha) e adicionar, se não tiver     46
Trocar o formato do vídeo para vertical                                                        45
Reduzir a duração do vídeo para até 15s (mediana na plataforma: ~10s)                          36
Name: n_campanhas, d

### 3.1 Distribuição do `delta_score` por ação, nas 500 campanhas

Para ver o quanto o efeito no score (what-if) varia entre campanhas — inclusive para ações que seriam descartadas das recomendações finais por terem impacto desprezível/negativo sem evidência de performance.

In [8]:
rec.distribuicao_delta_score(df)

,n,mediana,minimo,maximo
acao,,,,
Adicionar um hook nos primeiros 3 segundos,244,16.65,8.52,24.10
Reduzir a quantidade de texto na tela de muito para médio,91,13.07,4.79,19.86
Incluir rosto humano no vídeo,231,10.75,4.13,22.65
Adicionar legendas,191,7.53,1.01,16.57
Adicionar uma chamada para ação,153,7.36,1.66,13.19
Verificar se o vídeo tem legendas,46,4.51,-0.08,7.51
Trocar o formato do vídeo para vertical,231,1.38,-2.45,7.70
Ajustar a proporção música/voz para mais equilibrada,186,0.74,-9.03,13.47
Reduzir a duração do vídeo para até 15s,177,0.14,-6.00,6.27


Confirma o esperado (item 11): a duração do vídeo tem `delta_score` mediano próximo de 0 (às vezes negativo) — a importância ~0 encontrada na Parte 2 não é um acaso do conjunto de teste, é consistente nas 500 campanhas. Hook e rosto humano têm o maior `delta_score` mediano, de forma consistente (mínimo e máximo sempre positivos e relativamente altos) — reforça por que são os atributos mais importantes do modelo.